# Lesson 1 Demo Notebook: Hello Software 3.0 + 2026 Bonus

This notebook consolidates Lesson 1 into one live demo flow.

Run sections in order:
1. Setup
2. Chat completion
3. Streaming + TTFT
4. Voice pipeline (using a fixture audio file)
5. Image generation
6. Testing mindset quick check
7. Bonus (new): Responses API web search
8. Bonus (new): Server-side compaction


In [1]:
import os
import time
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY is missing. Add it to your .env file.")

client = OpenAI(api_key=api_key)
print("OpenAI client ready.")


OpenAI client ready.


## 1) Chat Completion

This is the core primitive: text in, text out.

In [2]:
prompt = "Explain Software 3.0 in one sentence."

start = time.time()
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.7,
)
elapsed = time.time() - start

print(response.choices[0].message.content)
print(f"\nElapsed: {elapsed:.2f}s")

# Format usage nicely
usage = response.usage
print("\n📊 Token Usage:")
print(f"  Total: {usage.total_tokens} tokens")
print(f"  Input: {usage.prompt_tokens} tokens")
print(f"  Output: {usage.completion_tokens} tokens")
if hasattr(usage, 'prompt_tokens_details') and usage.prompt_tokens_details.cached_tokens > 0:
    print(f"  Cached: {usage.prompt_tokens_details.cached_tokens} tokens")

Software 3.0 refers to a paradigm where software is primarily created by training machine learning models on data rather than being explicitly programmed with traditional code.

Elapsed: 1.53s

📊 Token Usage:
  Total: 47 tokens
  Input: 17 tokens
  Output: 30 tokens


### System messages

The `system` role lets you steer *how* the model responds — its tone, format, persona — without changing the user's question. Same input, different behavior.

In [3]:
question = "Explain Software 3.0 in one sentence."

# Without a system message
r1 = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": question}],
    temperature=0.7,
)
print("No system message:")
print(r1.choices[0].message.content)

# With a system message
r2 = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "system", "content": "You are a sarcastic comedian. Keep it short."},
        {"role": "user", "content": question},
    ],
    temperature=0.7,
)
print("\nWith system = 'sarcastic comedian':")
print(r2.choices[0].message.content)

No system message:
Software 3.0 refers to a paradigm where software development increasingly relies on training machine learning models on data rather than writing explicit code.

With system = 'sarcastic comedian':
Software 3.0 is when we stop writing code and just let AI guess what we want—because who needs humans making mistakes, right?


## 2) Streaming + TTFT

Streaming improves perceived latency by reducing time-to-first-token (TTFT).

In [4]:
start = time.time()
first_token_at = None
collected = []

stream = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": "Write a short haiku about APIs."}],
    stream=True,
)

print("Response: ", end="", flush=True)
for chunk in stream:
    content = chunk.choices[0].delta.content
    if not content:
        continue
    if first_token_at is None:
        first_token_at = time.time()
    collected.append(content)
    print(content, end="", flush=True)

total = time.time() - start
ttft = (first_token_at - start) if first_token_at else None

print("\n")
print(f"TTFT: {ttft:.2f}s" if ttft else "TTFT: n/a")
print(f"Time to full response: {total:.2f}s")


Response: Silent code connects,  
Bridges in the digital—  
APIs whisper.

TTFT: 0.64s
Time to full response: 0.81s


## 3) Voice Pipeline

Speech → Text (Whisper) → LLM → Text-to-Speech (TTS).

**Skipped for now** — uncomment the cell below to run live.

In [ ]:
# import pyaudio
# import wave
# import tempfile
# import os

# # Live microphone recording
# FORMAT = pyaudio.paInt16
# CHANNELS = 1
# RATE = 16000
# CHUNK = 1024
# RECORD_SECONDS = 3
# WAVE_OUTPUT_FILENAME = "temp_recording.wav"

# print("🎤 Recording for 3 seconds... Speak now!")

# audio = pyaudio.PyAudio()
# stream = audio.open(format=FORMAT, channels=CHANNELS,
#                     rate=RATE, input=True,
#                     frames_per_buffer=CHUNK)

# frames = []
# for i in range(0, int(RATE / CHUNK * RECORD_SECONDS)):
#     data = stream.read(CHUNK)
#     frames.append(data)

# print("✅ Recording finished!")
# stream.stop_stream()
# stream.close()
# audio.terminate()

# # Save temporary recording
# with wave.open(WAVE_OUTPUT_FILENAME, 'wb') as wf:
#     wf.setnchannels(CHANNELS)
#     wf.setsampwidth(audio.get_sample_size(FORMAT))
#     wf.setframerate(RATE)
#     wf.writeframes(b''.join(frames))

# try:
#     # Transcribe the recording
#     with open(WAVE_OUTPUT_FILENAME, "rb") as f:
#         transcript = client.audio.transcriptions.create(
#             model="whisper-1",
#             file=f,
#         )

#     user_text = transcript.text
#     print(f"🎯 You said: {user_text}")

#     # Get AI response
#     chat = client.chat.completions.create(
#         model="gpt-4.1-mini",
#         messages=[
#             {"role": "system", "content": "You are a helpful assistant. Keep responses brief."},
#             {"role": "user", "content": user_text},
#         ],
#     )
#     assistant_text = chat.choices[0].message.content
#     print(f"🤖 Assistant: {assistant_text}")

#     # Convert response to speech
#     speech = client.audio.speech.create(
#         model="tts-1",
#         voice="nova",
#         input=assistant_text,
#     )

#     out_path = Path("/tmp/l1_voice_reply.mp3")
#     speech.stream_to_file(out_path)
#     print(f"🔊 Saved TTS output: {out_path}")

#     # Play the response
#     try:
#         from IPython.display import Audio, display
#         print("🔊 Playing response...")
#         display(Audio(str(out_path), autoplay=True))
#     except Exception as e:
#         print(f"Audio playback not available: {e}")
#         print(f"Play the file manually: {out_path}")

# finally:
#     # Clean up temporary file
#     if os.path.exists(WAVE_OUTPUT_FILENAME):
#         os.remove(WAVE_OUTPUT_FILENAME)

## 4) Image Generation

DALL-E 3 image generation with revised prompt inspection.

**Skipped for now** — uncomment the cell below to run (~$0.04 per call).

In [ ]:
# img_prompt = "A robot teaching a classroom of humans about artificial intelligence at the University of Trento, italy. digital art style"

# start = time.time()
# img = client.images.generate(
#     model="dall-e-3",
#     prompt=img_prompt,
#     size="1024x1024",
#     quality="standard",
#     n=1,
# )
# elapsed = time.time() - start

# print(f"Image generated in {elapsed:.1f}s")
# print("Revised prompt:")
# print(img.data[0].revised_prompt)
# print("\nTemporary URL (~1 hour):")
# print(img.data[0].url)

## 5) Testing Mindset Quick Check

In traditional software, `sort([3,1,2])` must return `[1,2,3]`. One correct answer.

Now consider the prompt you already ran in Section 1: *"Explain Software 3.0 in one sentence."*

Run it three times — you'll get three different answers, all valid. No `assert ==` can capture this. The problem isn't formatting variation; it's that **"correct" is a spectrum, not a point**.

So what *can* we test? Structural checks, property bounds, and LLM-as-judge.

In [7]:
prompt = "Explain Software 3.0 in one sentence."

# Run the SAME prompt three times
for i in range(1, 4):
    r = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
    )
    answer = r.choices[0].message.content
    print(f"Run {i}: {answer}\n")

# All different, all correct. What would you assert?
# The best we can do: structural and property checks
assert answer is not None and len(answer) > 0, "Response should not be empty"
assert len(answer) < 500, "A one-sentence answer shouldn't be 500+ chars"
print("✓ Structural checks pass — but they don't tell us if the answer is *good*.")

Run 1: Software 3.0 refers to a paradigm where software is primarily created and improved through machine learning models that learn from data, rather than being explicitly programmed by humans.

Run 2: Software 3.0 refers to the paradigm where software is primarily written by training machine learning models on data, rather than being explicitly programmed by humans.

Run 3: Software 3.0 refers to the paradigm shift where software development increasingly relies on training machine learning models to automatically learn patterns from data, rather than explicitly programming rules by humans.

✓ Structural checks pass — but they don't tell us if the answer is *good*.


## 6) Bonus (New): Responses API + Web Search

### Chat Completions vs Responses API

Everything we've used so far is the **Chat Completions API** (`client.chat.completions.create`). It does one thing well: you send messages in, you get a message back. Stateless, simple, predictable.

The **Responses API** (`client.responses.create`) is OpenAI's newer primitive, designed for **agentic** use cases. Key differences:

| | Chat Completions | Responses API |
|---|---|---|
| **Mental model** | Single turn: messages in, message out | Multi-step: the model can take *actions* before responding |
| **Built-in tools** | None — you implement tool calls yourself | `web_search`, `file_search`, `code_interpreter` built in |
| **State** | You manage conversation history | Server can manage it (with `previous_response_id`) |
| **Context management** | You truncate/summarize manually | Server-side compaction available |

In short: Chat Completions is the low-level building block. Responses API is a higher-level interface where the model can autonomously search the web, run code, or read files *within a single API call* — before returning a final answer.

The example below uses `web_search` — the model decides to search, reads the results, and synthesizes an answer, all in one call.

Note: availability depends on account/project permissions and model.

In [ ]:
try:
    resp = client.responses.create(
        model="gpt-5",
        tools=[{"type": "web_search"}],
        tool_choice="auto",
        input="Find one positive AI news story from this week. Include source.",
    )
    print(resp.output_text)
except Exception as e:
    print("Web search demo unavailable in this project or region.")
    print(type(e).__name__, e)


## 7) Bonus (New): Server-side Compaction

As of Feb 2026, Responses API supports server-side compaction.

This is useful for long conversations to reduce context size and latency.

In [ ]:
long_text = "Summarize this text in 5 bullets: " + ("AI systems need good evaluation. " * 300)

try:
    compaction_resp = client.responses.create(
        model="gpt-5.2",
        input=[{"type": "message", "role": "user", "content": long_text}],
        store=False,
        context_management=[{"type": "compaction", "compact_threshold": 1200}],
    )

    print(compaction_resp.output_text[:1000])

    item_types = []
    for item in compaction_resp.output:
        item_type = getattr(item, "type", None)
        if item_type:
            item_types.append(item_type)

    print("\nOutput item types:", item_types)
    if "compaction" in item_types:
        print("Compaction item detected.")
    else:
        print("No compaction item detected in this run (threshold may not have been crossed).")
except Exception as e:
    print("Compaction demo unavailable in this project.")
    print(type(e).__name__, e)


## Next Step

For the in-class script version, use:
- `labs/01_hello_world/1_chat.py`
- `labs/01_hello_world/2_streaming.py`
- `labs/01_hello_world/3_voice.py`
- `labs/01_hello_world/4_image.py`